# NeuroSLM — Colab Training & Inference

This notebook clones the repository, installs dependencies, trains the 170M param model with QA datasets, and persists checkpoints via Git LFS so they survive runtime disconnects.

**Before running:** Runtime → Change runtime type → GPU

**Important:** Set your GitHub PAT in cell 2 so checkpoints can be pushed/pulled.

In [ ]:
# 1) Quick runtime / GPU check
import sys
print('Python', sys.version)

# Check GPU visibility
try:
    import subprocess; subprocess.run(['nvidia-smi'])
except Exception:
    print('nvidia-smi not available in shell; continue to check PyTorch')

import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('cuda device count:', torch.cuda.device_count())

In [ ]:
# 2) Clone repo with PAT for push access
# ⚠️ PASTE YOUR GITHUB TOKEN BELOW (or use Colab secrets)
GITHUB_TOKEN = ""  # <-- paste your token here, e.g. "ghp_xxxx..."

import os
if not GITHUB_TOKEN:
    from google.colab import userdata
    try:
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
        print("Token loaded from Colab secrets")
    except Exception:
        print("⚠️ No token set! Checkpoints won't be pushed to Git.")
        print("Set GITHUB_TOKEN above or add it to Colab secrets.")

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/269652/neuroslm.git' if GITHUB_TOKEN else 'https://github.com/269652/neuroslm.git'

if os.path.exists('neuroslm'):
    import shutil
    shutil.rmtree('neuroslm')

!git clone {REPO_URL}
%cd neuroslm
!git config user.email "train@neuroslm"
!git config user.name "NeuroSLM Trainer"

In [ ]:
# 3) Install dependencies + Git LFS + restore any existing checkpoints
!pip install -q --upgrade pip
!pip install -q -r requirements.txt || true
!apt-get install -y git-lfs -qq
!git lfs install

# Pull any previously saved LFS checkpoints
!git lfs pull
import glob
lfs_ckpts = sorted(glob.glob('lfs_checkpoints/*.pt'))
if lfs_ckpts:
    # Copy LFS checkpoints to working checkpoints dir
    import shutil, os
    os.makedirs('checkpoints', exist_ok=True)
    for c in lfs_ckpts:
        dest = os.path.join('checkpoints', os.path.basename(c))
        if not os.path.exists(dest):
            shutil.copy2(c, dest)
            print(f"Restored: {os.path.basename(c)}")
    print(f"✓ {len(lfs_ckpts)} checkpoint(s) restored from Git LFS")
else:
    print("No existing LFS checkpoints — will train from scratch")

In [ ]:
# 4) Optional: If PyTorch in the runtime is not CUDA-enabled, try a common CUDA wheel (cu121).
import torch
if not torch.cuda.is_available():
    print('PyTorch not CUDA-enabled. Attempting to install a CUDA wheel (cu121).')
    !python -m pip install --upgrade pip
    !python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 || true
    import importlib
    importlib.reload(__import__('torch'))
    print('after install: torch', __import__('torch').__version__)
    print('CUDA available now?', __import__('torch').cuda.is_available())
else:
    print('PyTorch already CUDA-enabled; good to go.')

In [ ]:
# 5) Train — 170M param model with QA datasets
# Checkpoints auto-pushed to Git LFS after training completes.
# If you already have a restored checkpoint, you can skip this cell.
import glob
existing = sorted(glob.glob('checkpoints/*.pt'))
if existing:
    print(f"Found existing checkpoint: {existing[-1]}")
    print("To retrain, delete checkpoints/ first. Skipping training.")
else:
    !python -m neuroslm.train --preset large --steps 3000 --batch_size 4 --device cuda --meta --mode mix --save_every 500

In [ ]:
# 6) Push checkpoint to Git LFS (manual backup — also happens auto after training)
import glob, shutil, os, subprocess

ckpts = sorted(glob.glob('checkpoints/*.pt'))
if ckpts:
    latest = ckpts[-1]
    os.makedirs('lfs_checkpoints', exist_ok=True)
    lfs_path = os.path.join('lfs_checkpoints', os.path.basename(latest))
    shutil.copy2(latest, lfs_path)
    print(f"Pushing {os.path.basename(latest)} to Git LFS...")
    !git add lfs_checkpoints/
    !git commit -m "chkpt: {os.path.basename(latest)}"
    !git push
    print("✓ Checkpoint saved to Git LFS — survives runtime disconnects!")
else:
    print("No checkpoints to push.")

In [ ]:
# 7) Generate text from the latest checkpoint
import glob
ckpts = sorted(glob.glob('checkpoints/*.pt'))
if ckpts:
    latest = ckpts[-1]
    print(f'Using checkpoint: {latest}')
    !python -m neuroslm.generate --ckpt {latest} --prompt "Explain how neural networks learn" --max_new 256 --temperature 0.7 --top_k 40 --device cuda
else:
    print('No checkpoints found. Run training first.')

In [ ]:
# 7) Virtual World Mind Wandering — let the brain dream in a simulated environment
import glob, torch, sys
sys.path.insert(0, '.')
from neuroslm.config import BrainConfig
from neuroslm.brain import Brain
from neuroslm.tokenizer import Tokenizer

# Load latest checkpoint
ckpts = sorted(glob.glob('checkpoints/*.pt'))
assert ckpts, "No checkpoints found. Train first."
latest = ckpts[-1]
print(f"Loading: {latest}")

tok = Tokenizer()
cfg = BrainConfig.large()
cfg.vocab_size = tok.vocab_size
brain = Brain(cfg).to("cuda")
ckpt = torch.load(latest, map_location="cuda", weights_only=False)
brain.load_state_dict(ckpt["model"], strict=False)
brain.eval()
print(f"Model loaded ({sum(p.numel() for p in brain.parameters())/1e6:.1f}M params)")

# Dream in each virtual environment
from neuroslm.environments.virtual_world import ENVIRONMENTS
state = brain.init_latents(1, "cuda")

for env_name in ["meadow_tree", "bus_stop", "campfire"]:
    print(f"\n{'='*60}")
    print(f"  DREAMING IN: {env_name}")
    print(f"{'='*60}")
    info = brain.dream(state, max_steps=10, environment=env_name, seed=42)
    print(f"  Consciousness: gamma={info['consciousness']['gamma']:.3f}, "
          f"phi={info['consciousness']['phi']:.3f}, "
          f"binding={info['consciousness']['binding']:.3f}")
    print(f"  Thought Transformer: consistency={info['thought_transformer']['consistency']:.3f}, "
          f"depth={info['thought_transformer']['depth']:.3f}")
    print(f"  Last sensory: {info['sensory_frame']['text'][:120]}...")
    print()

In [ ]:
# 8) Interactive Chat Console — talk with the model
import torch
from IPython.display import clear_output

def chat(brain, tok, device="cuda", max_new=200, temperature=0.75, top_k=40):
    """Interactive chat loop. Type 'quit' to exit, 'dream' to mind-wander."""
    state = brain.init_latents(1, device)
    history = []
    print("=" * 60)
    print("  NeuroSLM Interactive Chat")
    print("  Commands: 'quit' to exit, 'dream' to mind-wander,")
    print("            'reset' to clear history, 'status' for brain state")
    print("=" * 60)
    print()

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n[Session ended]")
            break

        if not user_input:
            continue
        if user_input.lower() == 'quit':
            print("[Goodbye]")
            break
        if user_input.lower() == 'reset':
            state = brain.init_latents(1, device)
            history = []
            print("[State reset]\n")
            continue
        if user_input.lower() == 'dream':
            import random
            env = random.choice(["meadow_tree", "bus_stop", "campfire", "ocean_cliff", "library", "rainy_window"])
            print(f"[Dreaming in {env}...]")
            info = brain.dream(state, max_steps=5, environment=env, seed=random.randint(0,9999))
            frame = info.get("sensory_frame", {})
            print(f"  {frame.get('text', '')[:150]}")
            print(f"  [consciousness: phi={info['consciousness']['phi']:.3f}]")
            print()
            continue
        if user_input.lower() == 'status':
            nt = {n: float(brain.transmitters.get(n).mean()) for n in brain.transmitters.names}
            print(f"  NT levels: {nt}")
            print(f"  Narrative: {brain.narrative_system.info()}")
            print()
            continue

        # Format as chat
        prompt = f"User: {user_input}\nAssistant:"
        history.append(prompt)
        full_prompt = "\n".join(history[-6:])  # keep last 3 exchanges

        ids = torch.tensor([tok.encode(full_prompt)], dtype=torch.long, device=device)
        ids = ids[:, -brain.cfg.lang_ctx:]

        with torch.no_grad():
            out_ids = brain.generate(
                ids, max_new=max_new, temperature=temperature,
                top_k=top_k, use_convergent=True)

        response = tok.decode(out_ids[0].tolist())
        # Extract assistant response
        if "Assistant:" in response:
            response = response.split("Assistant:")[-1].strip()
        # Truncate at next "User:" if present
        if "User:" in response:
            response = response.split("User:")[0].strip()

        print(f"Assistant: {response}\n")
        history.append(f"Assistant: {response}")

# Run chat (model should already be loaded from cell 7, or load here)
try:
    brain
except NameError:
    import glob
    from neuroslm.config import BrainConfig
    from neuroslm.brain import Brain
    from neuroslm.tokenizer import Tokenizer
    tok = Tokenizer()
    cfg = BrainConfig.large()
    cfg.vocab_size = tok.vocab_size
    brain = Brain(cfg).to("cuda")
    ckpts = sorted(glob.glob('checkpoints/*.pt'))
    ckpt = torch.load(ckpts[-1], map_location="cuda", weights_only=False)
    brain.load_state_dict(ckpt["model"], strict=False)
    brain.eval()

chat(brain, tok)

## Troubleshooting & notes
- If your repo is private, clone with a PAT: `git clone https://<PAT>@github.com/<owner>/neuroslm.git`
- If LFS objects are missing, ensure you ran `git lfs pull` and you have access to the LFS storage.
- If the runtime's PyTorch wheel mismatches GPU drivers, try a different wheel tag (cu118, cu121). Check `nvidia-smi` output.
- To run multiple training iterations, increase `--steps` and `--batch_size` (be mindful of Colab memory limits).

If you want, I can also add PAT-safe entry cells (to pass a token via a form) or mount Google Drive instead of cloning.